<a href="https://colab.research.google.com/github/JoyNgaru/Air-Quality-Indexing/blob/main/Air_Quality_Indexing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Loading and Machine Learning Workflow

This notebook will guide you through loading your air quality data from Google Drive and applying a machine learning workflow including data preprocessing, PCA for dimensionality reduction, and a Random Forest classifier for prediction.

### 1. Mount Google Drive

To access your CSV file, we first need to mount your Google Drive. Please follow the prompts to authorize Colab to access your Drive.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Load the Dataset

Now that your Drive is mounted, we can load the `device2_air_quality_2024-09-25_to-2025-04-07.csv` file into a pandas DataFrame. Make sure this file is located in your Google Drive, and adjust the `file_path` if it's in a specific subfolder within `MyDrive`.

In [15]:
import pandas as pd

# Define the path to your CSV file on Google Drive
# Adjust the path if your file is in a subfolder, e.g., '/content/drive/MyDrive/my_data/device2_air_quality_2024-09-25_to_2025-04-07.csv'
file_path = '/content/drive/MyDrive/device2_air_quality_2024-09-25_to_2025-04-07.csv'

try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the file is in your Google Drive and the path is correct.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Dataset loaded successfully!


,Timestamp,deviceId,payload_id,air_temperature,humidity,CO2,pm1,pm10,pm2_5,air_pressure
0,2024-09-25 19:25:00,2,1,26.30,64.26,455,61.5,119.9,111.1,100994
1,2024-09-25 19:30:00,2,2,26.29,64.29,452,60.2,119.8,111.0,100995
2,2024-09-25 19:35:00,2,3,26.30,64.32,442,63.1,125.5,114.3,100998
3,2024-09-25 19:40:00,2,4,26.29,64.34,442,62.1,123.0,112.9,100998
4,2024-09-25 19:45:00,2,5,26.29,64.44,437,62.7,124.2,114.4,101000


### 2.1 Create AQI Category Target Variable

Since the original dataset does not contain an 'AQI_category', we will create one based on the 'pm2_5' (PM2.5 concentration) values. This is a common practice to define air quality levels for classification tasks. We will use the following breakpoints, inspired by EPA standards:

*   **Good**: PM2.5 <= 12.0
*   **Moderate**: 12.1 <= PM2.5 <= 35.4
*   **Unhealthy for Sensitive Groups**: 35.5 <= PM2.5 <= 55.4
*   **Unhealthy**: 55.5 <= PM2.5 <= 150.4
*   **Very Unhealthy**: 150.5 <= PM2.5 <= 250.4
*   **Hazardous**: PM2.5 > 250.4

In [16]:
import numpy as np

if 'df' in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    if 'pm2_5' in df.columns:
        # Define AQI categories based on PM2.5 values (EPA breakpoints)
        bins = [
            0.0,
            12.0,
            35.4,
            55.4,
            150.4,
            250.4,
            np.inf
        ]
        labels = [
            'Good',
            'Moderate',
            'Unhealthy for Sensitive Groups',
            'Unhealthy',
            'Very Unhealthy',
            'Hazardous'
        ]

        df['AQI_category'] = pd.cut(df['pm2_5'], bins=bins, labels=labels, right=True, include_lowest=True)
        print("Successfully created 'AQI_category' column based on PM2.5 levels.")
        print("Distribution of new AQI categories:")
        display(df['AQI_category'].value_counts())
        display(df.head())
    else:
        print("Error: 'pm2_5' column not found in the DataFrame. Cannot create 'AQI_category'.")
else:
    print("DataFrame 'df' is not loaded or is empty. Please ensure the data loading step was executed successfully.")


Successfully created 'AQI_category' column based on PM2.5 levels.
Distribution of new AQI categories:


,count
AQI_category,
Unhealthy,23811
Unhealthy for Sensitive Groups,15115
Moderate,6397
Very Unhealthy,6011
Hazardous,615
Good,310


,Timestamp,deviceId,payload_id,air_temperature,humidity,CO2,pm1,pm10,pm2_5,air_pressure,AQI_category
0,2024-09-25 19:25:00,2,1,26.30,64.26,455,61.5,119.9,111.1,100994,Unhealthy
1,2024-09-25 19:30:00,2,2,26.29,64.29,452,60.2,119.8,111.0,100995,Unhealthy
2,2024-09-25 19:35:00,2,3,26.30,64.32,442,63.1,125.5,114.3,100998,Unhealthy
3,2024-09-25 19:40:00,2,4,26.29,64.34,442,62.1,123.0,112.9,100998,Unhealthy
4,2024-09-25 19:45:00,2,5,26.29,64.44,437,62.7,124.2,114.4,101000,Unhealthy


### 3. Data Preprocessing and Model Training

This section implements the machine learning workflow you provided:

1.  **Select Features**: Defines the features (`PM2.5`, `PM10`, `CO2`, `NO2`, `temperature`, `humidity`) and the target variable (`AQI_category`).
2.  **Clean Missing Values**: Removes rows with any missing values from the selected features.
3.  **Scale Features**: Standardizes the feature data using `StandardScaler`.
4.  **PCA**: Applies Principal Component Analysis to reduce dimensionality while retaining 95% of the variance.
5.  **Train-Test Split**: Divides the data into training and testing sets.
6.  **Random Forest Classifier**: Trains a `RandomForestClassifier` model.
7.  **Predictions**: Makes predictions on the test set.
8.  **Evaluation**: Prints the confusion matrix and classification report to evaluate the model's performance.
9.  **Feature Importance**: Attempts to show feature importances (note: interpretation of feature importance after PCA can be complex as it relates to the principal components, not original features directly).

In [17]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Ensure df is loaded before proceeding
# Using globals() for robustness in interactive environments
if 'df' in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    # 2. Select features (example pollutants + weather)
    # Corrected feature names based on df.head() and removed missing 'NO2'
    features = ['pm2_5','pm10','CO2','air_temperature','humidity']

    # Check if all features exist in the DataFrame
    missing_features = [f for f in features if f not in df.columns]
    if missing_features:
        print(f"Error: The following features are missing from the DataFrame: {missing_features}. Please check your CSV file or adjust the 'features' list.")
    else:
        X = df[features]

        # Target: AQI category (must be pre-labeled in dataset)
        # Check if 'AQI_category' exists
        if 'AQI_category' not in df.columns:
            # Inform the user that AQI_category is missing and how to proceed
            print("Error: 'AQI_category' column not found in the DataFrame. ")
            print("Please ensure your CSV contains this column for classification, or define it based on other columns (e.g., creating categories from PM2.5 levels).")
            print("The machine learning pipeline cannot proceed without a target variable.")
        else:
            y = df['AQI_category']   # e.g., Good, Moderate, Unhealthy

            # 3. Clean missing values
            initial_rows = len(X)
            X = X.dropna()
            y = y.loc[X.index]
            if len(X) < initial_rows:
                print(f"Removed {initial_rows - len(X)} rows due to missing values.")
            else:
                print("No missing values found in selected features.")

            if not X.empty:
                # 4. Scale features
                scaler = StandardScaler()
                X_scaled = scaler.fit_transform(X)

                # 5. PCA
                # Check if enough samples for PCA after scaling
                if X_scaled.shape[0] > 1:
                    pca = PCA(n_components=0.95)  # keep 95% variance
                    X_pca = pca.fit_transform(X_scaled)

                    print("Explained variance ratio:", pca.explained_variance_ratio_)
                    print(f"Number of components after PCA: {pca.n_components_}")

                    # 6. Train-test split
                    # Check if there's enough data for splitting after cleaning
                    if len(X_pca) > 1: # At least 2 samples for train-test split
                        # Ensure there are enough classes for stratification if needed, though default split is not stratified.
                        # Also, ensure y has enough unique values for classification.
                        if len(y.unique()) > 1:
                            X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

                            # 7. Random Forest
                            rf = RandomForestClassifier(n_estimators=100, random_state=42)
                            rf.fit(X_train, y_train)

                            # 8. Predictions
                            y_pred = rf.predict(X_test)

                            # 9. Evaluation
                            print("\nConfusion Matrix:")
                            print(confusion_matrix(y_test, y_pred))
                            print("\nClassification Report:")
                            print(classification_report(y_test, y_pred))

                            # 10. Feature importance (from PCA components)
                            # Note: Feature importances from Random Forest directly relate to the PCA components, not the original features.
                            # To get original feature importance, one would typically use feature importance before PCA or interpret PCA components.
                            importances = rf.feature_importances_
                            print("\nRandom Forest feature importances (for PCA components):", importances)
                        else:
                            print("Error: Target variable 'AQI_category' has only one unique value after cleaning. Cannot perform classification.")
                    else:
                        print("Not enough data remaining after preprocessing for train-test split.")
                else:
                    print("Not enough samples for PCA after data preprocessing.")
            else:
                print("DataFrame is empty after cleaning missing values. Cannot proceed with PCA and model training.")
else:
    print("DataFrame 'df' is not loaded or is empty. Please ensure the data loading step in the previous cell was executed successfully.")

No missing values found in selected features.
Explained variance ratio: [0.4620904  0.26476655 0.2025103  0.07022594]
Number of components after PCA: 4

Confusion Matrix:
[[  62    0    3    0    0    0]
 [   0  117    0    0    0    4]
 [   2    0 1270    0   29    0]
 [   0    0    0 4742   24   15]
 [   0    0   22   41 2906    0]
 [   0    5    0   13    0 1197]]

Classification Report:
                                precision    recall  f1-score   support

                          Good       0.97      0.95      0.96        65
                     Hazardous       0.96      0.97      0.96       121
                      Moderate       0.98      0.98      0.98      1301
                     Unhealthy       0.99      0.99      0.99      4781
Unhealthy for Sensitive Groups       0.98      0.98      0.98      2969
                Very Unhealthy       0.98      0.99      0.98      1215

                      accuracy                           0.98     10452
                     macro a